# 07 — Libdevice

**Ladder:** Naive `math.erf` loop → vectorized `torch.erf` → Triton `libdevice.erf` → accuracy/perf tradeoff vs pure-`tl` tanh / polynomials.

**Study tip.** After running cells: write erf-GELU; say when a tanh poly beats libdevice; note that huge-$N$ GELU is usually memory-bound (accuracy often "free").

Official: [Extern functions / libdevice](https://triton-lang.org/main/getting-started/tutorials/07-extern-functions.html)
· [Libdevice slug](https://triton-lang.org/main/getting-started/tutorials/07-libdevice.html)

Figures: [`docs/images/`](docs/images/) · visual guide: [`docs/visual-guide.md`](docs/visual-guide.md)

![Whiteboard: pure Triton approx vs libdevice device math](docs/images/benchmark-libdevice.png)

*Theory board — polynomial/tanh approximations vs `tl.extra.libdevice` intrinsics (accuracy / speed tradeoff).*

---

# Part I — Mathematics

## 0. GELU: definition, derivatives, and approximations

### 0.1 The error function

The **error function** is

$$
\mathrm{erf}(z)=\frac{2}{\sqrt{\pi}}\int_0^{z}e^{-t^{2}}\,\mathrm{d}t .
\tag{0.1}
$$

It is odd, bounded, and satisfies

$$
\mathrm{erf}(0)=0,
\qquad
\lim_{z\to\pm\infty}\mathrm{erf}(z)=\pm1,
\qquad
\frac{\mathrm{d}}{\mathrm{d}z}\mathrm{erf}(z)
=
\frac{2}{\sqrt{\pi}}e^{-z^{2}} .
\tag{0.2}
$$

The standard normal CDF is

$$
\Phi(x)=\frac{1}{\sqrt{2\pi}}\int_{-\infty}^{x}e^{-t^{2}/2}\,\mathrm{d}t
=
\frac12\left(1+\mathrm{erf}\!\left(\frac{x}{\sqrt2}\right)\right).
\tag{0.3}
$$

*Proof.* Substitute $t=z/\sqrt2$ in (0.1) and use
$\int_{-\infty}^{0}e^{-t^{2}/2}dt=\sqrt{\pi/2}$. $\square$

### 0.2 Definition of GELU

The Gaussian Error Linear Unit is

$$
\mathrm{GELU}(x)
=
x\,\Phi(x)
=
\frac{x}{2}\left(1+\mathrm{erf}\!\left(\frac{x}{\sqrt2}\right)\right).
\tag{0.4}
$$

**Interpretation.** $\Phi(x)$ is the probability that a standard normal variable
is at most $x$; GELU weights the input by its own percentile. Unlike ReLU, it is
smooth and can be negative for $x<0$ (with small magnitude).

**Asymptotics.**

$$
\mathrm{GELU}(x)\sim
\begin{cases}
x, & x\to+\infty,\\
0, & x\to-\infty,
\end{cases}
\qquad
\mathrm{GELU}(0)=0.
\tag{0.5}
$$

**Proposition 1 (derivative).** With $\varphi(x)=\frac{1}{\sqrt{2\pi}}e^{-x^{2}/2}$
the standard normal density,

$$
\mathrm{GELU}'(x)=\Phi(x)+x\,\varphi(x).
\tag{0.6}
$$

*Proof.* Differentiate $x\Phi(x)$:

$$
\frac{\mathrm{d}}{\mathrm{d}x}\big(x\Phi(x)\big)
=
\Phi(x)+x\Phi'(x)
=
\Phi(x)+x\varphi(x).
\qquad\square
$$

In the erf form, the same result reads

$$
\mathrm{GELU}'(x)
=
\frac12\left(1+\mathrm{erf}\!\left(\frac{x}{\sqrt2}\right)\right)
+
\frac{x}{\sqrt{2\pi}}e^{-x^{2}/2}.
\tag{0.7}
$$

The derivative is bounded and positive; GELU is monotone increasing (a useful
property for optimization).

### 0.3 The tanh approximation

A widely used closed-form approximation is

$$
\widehat{\mathrm{GELU}}_{\tanh}(x)
=
\frac{x}{2}\left(
1+\tanh\!\left(
\sqrt{\frac{2}{\pi}}\,\big(x+0.044715\,x^{3}\big)
\right)
\right).
\tag{0.8}
$$

**Where it comes from.** Use

$$
\Phi(x)\approx\frac12\left(1+\tanh\!\left(\sqrt{\frac{2}{\pi}}\,x\right)\right),
\tag{0.9}
$$

which matches the first terms of the Taylor expansion of both functions, then add
the cubic term $0.044715x^{3}$ to fit the tails. The constant was chosen
empirically to minimize the maximum absolute error on a representative range.

**Error.** On $\mathbb{R}$ the maximum absolute error of (0.8) is approximately
$4.7\times10^{-4}$ (at $|x|\approx2.7$), with relative error small for
$|x|\gtrsim1$. That is of the order of one FP16 ulp near $1$
($2^{-11}\approx4.9\times10^{-4}$), so for FP16 workloads the approximation is
often close to the exact GELU — but it is not far below FP16 resolution.

### 0.4 The sigmoid approximation

An even cheaper approximation is

$$
\widehat{\mathrm{GELU}}_{\sigma}(x)
=
x\,\sigma(1.702\,x),
\qquad
\sigma(z)=\frac{1}{1+e^{-z}}.
\tag{0.10}
$$

It is monotone and very cheap, but its maximum absolute error is about
$2\times10^{-2}$ — acceptable only for inference in low precision.

### 0.5 Numerical accuracy: ULP and rounding

A floating-point format with $p$ significand bits has unit roundoff

$$
u=2^{-p},
\qquad
\text{FP32}: p=24,\ u\approx5.96\times10^{-8},
\qquad
\text{FP16}: p=11,\ u\approx4.88\times10^{-4}.
\tag{0.11}
$$

Two implementations agree to $k$ **ULPs** if their values differ by at most
$k$ units in the last place. `libdevice.erf` is a device-math implementation of
the correctly-rounded error function to within a few ULPs; a tanh approximation
is not correctly rounded and its error is measured in absolute terms.

**Guard against catastrophic cancellation.** The formula
$\frac12(1+\mathrm{erf}(x/\sqrt2))$ loses relative accuracy for large negative
$x$, because $1+\mathrm{erf}(\cdot)\to0$ is a subtraction of nearly equal
numbers. Implementations use the complementary error function
$\mathrm{erfc}(z)=1-\mathrm{erf}(z)$ there:

$$
\Phi(x)=
\begin{cases}
\frac12\big(1+\mathrm{erf}(x/\sqrt2)\big), & x\ge0,\\[2pt]
\frac12\,\mathrm{erfc}(-x/\sqrt2), & x<0.
\end{cases}
\tag{0.12}
$$

This preserves relative accuracy in the left tail.

### 0.6 Is accuracy free?

For an elementwise kernel the arithmetic intensity is

$$
\mathrm{AI}
=
\frac{F}{Q}
\approx
\frac{c}{2b}
\ \text{FLOP/byte},
\tag{0.13}
$$

where $c$ is the number of FLOPs per element (roughly 5–20 depending on the
approximation) and $2b$ accounts for one load and one store. For FP32,
$\mathrm{AI}\approx1$–$3$ FLOP/byte, far below machine balance. Hence the kernel
is **memory-bound** and the cost of a more accurate special function is hidden
behind the memory latency — accuracy is effectively free as long as the extra
instructions do not lower occupancy. That is the practical conclusion of this
notebook.

### Theory board

![Whiteboard: pure Triton approx vs libdevice device math](docs/images/whiteboard-theory.jpg)

*Theory board — polynomial/tanh approximations vs `tl.extra.libdevice` intrinsics (accuracy / speed tradeoff).*

# Part II — Proof

**Theorem (GELU identity).** With
$\Phi(x)=\frac12(1+\mathrm{erf}(x/\sqrt2))$,
$\mathrm{GELU}(x)=x\Phi(x)$ and
$\mathrm{GELU}'(x)=\Phi(x)+x\varphi(x)$, where
$\varphi(x)=\frac1{\sqrt{2\pi}}e^{-x^2/2}$.

*Proof.* The first identity is the definition of $\Phi$; differentiating
$x\Phi(x)$ and using $\Phi'=\varphi$ gives the second. $\square$

**Proposition (tanh approximation).** The function (0.8) satisfies

$$
\sup_x|\widehat{\mathrm{GELU}}_{\tanh}(x)-\mathrm{GELU}(x)|
\lesssim 3\times10^{-4}.
$$

*Proof.* The approximation matches the Taylor expansion of
$\Phi$ through the linear term and the cubic term is fitted to minimize the
maximum error; a direct numerical maximization of the difference on a fine grid
gives the stated bound. $\square$

**Proposition (libdevice equivalence).** `libdevice.erf` evaluates the same
mathematical error function as `math.erf`, up to a few ULPs.

*Proof.* Both implement the same special function; the device library is a
correctly-rounded-to-few-ULP implementation. $\square$

# Part III — Step-by-step proof

1. **Define the map.** $y_i=\frac12x_i(1+\mathrm{erf}(x_i/\sqrt2))$.
2. **Naive evaluation.** Loop over $i$ and call the host `math.erf`.
3. **Vectorize.** Replace the loop by `torch.erf`, which is the same formula on
   a tensor.
4. **Kernel map.** In Triton, each program owns a masked tile of $B$ elements.
5. **Load.** Read $x_i$ and mask padded lanes.
6. **Special function.** Evaluate `libdevice.erf(x * 0.7071...)` on the tile.
7. **Affine combination.** Compute $0.5x(1+\mathrm{erf})$.
8. **Store.** Write masked outputs.
9. **Approximate path.** Replace step 6 by
   $\tanh(\sqrt{2/\pi}(x+0.044715x^3))$ to trade $\sim3\times10^{-4}$ error for
   fewer special-function instructions.
10. **Roofline.** The pass moves $2Nb$ bytes and does $O(N)$ work, so it is
    memory-bound; the special-function choice is usually hidden.

# Part IV — Coding

The coding part follows the three-rung ladder used in the PDFs: **Method 01 — naive**, **Method 02 — kernel-based**, and **Method 03 — fused kernel-based**. Each method is runnable and is checked against the previous one.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import math

import pytest
import torch
import triton
import triton.language as tl
import triton.language.extra.libdevice as libdevice

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")



### Method 01 — Naive (Python)

For each index $i$:

$$
y_i=\frac12x_i\left(1+\mathrm{erf}\!\left(\frac{x_i}{\sqrt2}\right)\right).
\tag{3.1}
$$

| Stage | Realization | Notes |
|---|---|---|
| **Naive** | Python loop + `math.erf` | Host reference; obvious formula |
| **Intermediate** | `torch.erf` vectorized | Same math; ATen, not custom Triton |
| **Triton** | `tl.load` → `libdevice.erf` → store | Device special function inside SPMD tile |

**Equivalence.** All three implement the same mathematical map (up to
floating-point rounding). Tests compare against `torch.nn.functional.gelu`.

### 3.1 SPMD tile

Program $p$ owns the offsets

$$
\mathrm{offsets}_p=pB+\{0,1,\dots,B-1\},
\qquad
m=\mathrm{offsets}_p<N,
\tag{3.2}
$$

exactly as in notebook 01. The kernel is a **pure map**: no reduction, no
cross-lane communication, no dependence between elements. Therefore:

- the grid is embarrassingly parallel;
- any block size is correct (masking handles the remainder);
- the only tuning knobs are vector width and occupancy.

**Pointer arithmetic.**

$$
\&x_i=\&x_0+i\,b,
\qquad
y_i=f\!\left(x_i\right),
\qquad
\&y_i=\&y_0+i\,b .
\tag{3.3}
$$

### 3.2 What `libdevice.erf` is

`triton.language.extra.libdevice` exposes the CUDA device math library. The call
`libdevice.erf(x)` lowers to the PTX/NVVM intrinsic for the appropriate type
(`__nv_erff` for FP32, `__nv_erf` for FP64). It is not a Triton-level
approximation: it is the vendor's tuned implementation, typically a
piecewise polynomial/rational approximation with range reduction.

**Dispatch rules.**

- FP32 input → FP32 `erf`, typically a few ULPs;
- FP64 input → FP64 `erf`, much slower on consumer GPUs (1/32 or 1/64 rate);
- FP16 input → upcast to FP32, evaluate, cast back (the kernel does this
  explicitly).

### 3.3 Accuracy versus cost

Let $c_{\mathrm{erf}}$ and $c_{\tanh}$ be the instruction counts per element of the
two paths, and let $P_{\mathrm{alg}}$ be the rate at which the machine executes
them (elements per second). The measured runtime is

$$
t\approx L+\frac{2Nb}{\mathrm{BW}}
+\mathbf{1}[\text{compute-bound}]\cdot
\frac{N\,\bigl(c_{\mathrm{erf}}-c_{\tanh}\bigr)}{P_{\mathrm{alg}}} .
\tag{3.4}
$$

For large $N$ the memory term dominates and
$t_{\mathrm{erf}}\approx t_{\tanh}$. For small $N$ the launch term $L$
dominates. Only in the intermediate
regime can the special-function cost matter, and even then it is usually hidden
by latency.

**When to prefer libdevice vs pure Triton.**

| Situation | Choice |
|---|---|
| Need `erf`/`erfc`/`exp`/`log` with known accuracy | `libdevice` |
| Published approximation acceptable (tanh GELU) | pure `tl` |
| Very large $N$, memory-bound | either; accuracy is free |
| Register/occupancy limited | pure `tl` (fewer live values) |

### 3.4 Numerical check

The reference test compares

$$
\big|\hat y_i-y_i\big|\le\varepsilon_{\mathrm{abs}}
+\varepsilon_{\mathrm{rel}}|y_i|,
\tag{3.5}
$$

with $\varepsilon_{\mathrm{abs}}=\varepsilon_{\mathrm{rel}}=10^{-2}$ on GPU
(FP32 inputs) and $10^{-5}$ on CPU. The GPU tolerance is loose because the
libdevice `erf` path and `F.gelu` may use slightly different implementations; the
kernel itself accumulates and stores in FP32.

### 3.5 Composition with the rest of the kernel

GELU is typically fused into an MLP epilogue:

$$
h=\mathrm{GELU}(xW_1+b_1),
\qquad
y=hW_2+b_2 .
\tag{3.6}
$$

If the GEMM writes $xW_1+b_1$ to HBM and a separate GELU kernel reads it back,
the elementwise pass costs an extra $2Nb$ bytes. Fusing GELU into the GEMM
epilogue removes that round trip entirely — the same fusion principle as
notebooks 02 and 05, applied to an elementwise nonlinearity. The `libdevice`
call is cheap relative to the saved memory traffic.


In [ ]:
def naive_gelu_pseudo(x):
    """
    Exact GELU (Gaussian Error Linear Unit):

        gelu(x) = 0.5 * x * (1 + erf(x / sqrt(2)))

    Implemented with a Python loop over elements using math.erf so the
    formula is obvious. Prefer gelu_torch / the Triton kernel for speed.
    """
    flat = x.detach().cpu().reshape(-1)
    out = torch.empty_like(flat)
    inv_sqrt2 = 1.0 / math.sqrt(2.0)
    for i in range(flat.numel()):
        v = float(flat[i])
        out[i] = 0.5 * v * (1.0 + math.erf(v * inv_sqrt2))
    return out.reshape(x.shape).to(device=x.device, dtype=x.dtype)


In [ ]:
def gelu_torch(x):
    """Same formula with vectorized torch ops (still not a custom kernel)."""
    return 0.5 * x * (1.0 + torch.erf(x / math.sqrt(2.0)))


### Method 02 — Kernel-based (tanh approximation)

This kernel replaces the special function by the closed-form tanh approximation

$$
\widehat{\mathrm{GELU}}_{\tanh}(x)
=
\frac{x}{2}\left(1+\tanh\!\left(\sqrt{\frac{2}{\pi}}
\left(x+0.044715x^3\right)\right)\right).
$$

It is pure Triton arithmetic (plus `tanh`) and is the fast, approximate
kernel-based method.


In [ ]:
@triton.jit
def gelu_tanh_kernel(x_ptr, y_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    """Method 02: tanh approximation, pure Triton arithmetic."""
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask).to(tl.float32)
    # sqrt(2/pi)
    c = 0.7978845608028654
    inner = c * (x + 0.044715 * x * x * x)
    t = libdevice.tanh(inner)
    y = 0.5 * x * (1.0 + t)
    tl.store(y_ptr + offsets, y, mask=mask)


def gelu_tanh(x):
    y = torch.empty_like(x)
    n = x.numel()
    grid = lambda meta: (triton.cdiv(n, meta["BLOCK_SIZE"]),)
    gelu_tanh_kernel[grid](x, y, n, BLOCK_SIZE=1024)
    return y


### Method 03 — Fused kernel-based (`libdevice.erf`)

This is the accurate path: the same elementwise kernel, but the special
function is the device-math `libdevice.erf`. The load, erf, affine combination,
and store are fused in one pass; for large $N$ the extra accuracy is hidden by
the memory bound.


In [ ]:
@triton.jit
def gelu_kernel(x_ptr, y_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    """Elementwise GELU using CUDA libdevice erf via Triton's extra bindings."""
    pid = tl.program_id(0)
    offs = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask).to(tl.float32)
    # 0.7071... = 1/sqrt(2); libdevice.erf is the device math library
    y = libdevice.erf(x * 0.7071067811865476)
    y = 0.5 * x * (1.0 + y)
    tl.store(y_ptr + offs, y, mask=mask)


def gelu_libdevice(x):
    y = torch.empty_like(x)
    n = x.numel()
    grid = lambda meta: (triton.cdiv(n, meta["BLOCK_SIZE"]),)
    gelu_kernel[grid](x, y, n, BLOCK_SIZE=1024)
    return y


### Pitfalls

- `libdevice` availability differs by backend; check imports.
- Comparing tanh-GELU to erf-GELU needs looser tolerances.
- Special functions can use more registers → lower occupancy.
- Evaluating $1+\mathrm{erf}(z)$ for very negative $z$ loses relative precision.

### Self-check

1. Write GELU with `erf` and derive its derivative.
2. When is a pure Triton polynomial "good enough"?
3. Is a more accurate erf free if the kernel is memory-bound?
4. Estimate the max error of the tanh approximation in FP16 units.


In [ ]:
def test_pseudo_matches_torch() -> None:
    torch.manual_seed(0)
    x = torch.randn(128)
    ref = torch.nn.functional.gelu(x)
    assert torch.allclose(naive_gelu_pseudo(x), ref, atol=1e-5, rtol=1e-5)
    assert torch.allclose(gelu_torch(x), ref, atol=1e-5, rtol=1e-5)


def test_correctness() -> None:
    require_gpu()
    x = torch.randn(8192, device=DEVICE, dtype=torch.float32)
    y = gelu_libdevice(x)
    y_ref = torch.nn.functional.gelu(x)
    assert torch.allclose(y, y_ref, atol=1e-2, rtol=1e-2)
    print("gelu via libdevice OK")


In [ ]:
def test_gelu_kernel_methods() -> None:
    """Method 02 (tanh approximation) and Method 03 (libdevice erf)."""
    require_gpu()
    torch.manual_seed(0)
    x = torch.randn(8192, device=DEVICE, dtype=torch.float32)
    ref = torch.nn.functional.gelu(x)
    y_approx = gelu_tanh(x)
    y_exact = gelu_libdevice(x)
    assert torch.allclose(y_approx, ref, atol=1e-2, rtol=1e-2)
    assert torch.allclose(y_exact, ref, atol=1e-2, rtol=1e-2)

    # The exact path should be closer than the tanh approximation.
    err_approx = (y_approx - ref).abs().max().item()
    err_exact = (y_exact - ref).abs().max().item()
    assert err_exact <= err_approx, (err_exact, err_approx)
    print(f"GELU max error: tanh={err_approx:.3e}, libdevice={err_exact:.3e}")
